In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [26]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [27]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.10374219715595245
epoch:  1, loss: 0.07163327187299728
epoch:  2, loss: 0.05480676516890526
epoch:  3, loss: 0.04587159678339958
epoch:  4, loss: 0.041121385991573334
epoch:  5, loss: 0.03859865665435791
epoch:  6, loss: 0.03726138919591904
epoch:  7, loss: 0.03655387461185455
epoch:  8, loss: 0.03617994859814644
epoch:  9, loss: 0.03598235547542572
epoch:  10, loss: 0.035877879709005356
epoch:  11, loss: 0.03582247719168663
epoch:  12, loss: 0.03579293563961983
epoch:  13, loss: 0.03577696904540062
epoch:  14, loss: 0.03576817363500595
epoch:  15, loss: 0.035763151943683624
epoch:  16, loss: 0.035760097205638885
epoch:  17, loss: 0.035756297409534454
epoch:  18, loss: 0.035751838237047195
epoch:  19, loss: 0.035750612616539
epoch:  20, loss: 0.03574397414922714
epoch:  21, loss: 0.03574005141854286
epoch:  22, loss: 0.03573780134320259
epoch:  23, loss: 0.03573203831911087
epoch:  24, loss: 0.035728614777326584
epoch:  25, loss: 0.035725828260183334
epoch:  26, loss

In [28]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6981754302978516
Test metrics:  R2 = 0.6879717111587524


In [29]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="scaled",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.43126216530799866
epoch:  1, loss: 0.23494121432304382
epoch:  2, loss: 0.0717485323548317
epoch:  3, loss: 0.05228767171502113
epoch:  4, loss: 0.03688377887010574
epoch:  5, loss: 0.035281673073768616
epoch:  6, loss: 0.03509427234530449
epoch:  7, loss: 0.034959763288497925
epoch:  8, loss: 0.03475284203886986
epoch:  9, loss: 0.03474444895982742
epoch:  10, loss: 0.0345945730805397
epoch:  11, loss: 0.034548867493867874
epoch:  12, loss: 0.03440465033054352
epoch:  13, loss: 0.03427863493561745
epoch:  14, loss: 0.034164462238550186
epoch:  15, loss: 0.03404476121068001
epoch:  16, loss: 0.0339265801012516
epoch:  17, loss: 0.03378293290734291
epoch:  18, loss: 0.03367553651332855
epoch:  19, loss: 0.033579304814338684
epoch:  20, loss: 0.03345973789691925
epoch:  21, loss: 0.03331580385565758
epoch:  22, loss: 0.03320284187793732
epoch:  23, loss: 0.03308665752410889
epoch:  24, loss: 0.03296749293804169
epoch:  25, loss: 0.03283606469631195
epoch:  26, loss: 0.

In [30]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6951719522476196
Test metrics:  R2 = 0.6974952816963196


In [31]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    lr_init="BB1",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.23248465359210968
epoch:  1, loss: 0.13422323763370514
epoch:  2, loss: 0.08634088188409805
epoch:  3, loss: 0.06182826682925224
epoch:  4, loss: 0.049169328063726425
epoch:  5, loss: 0.042628806084394455
epoch:  6, loss: 0.03925637528300285
epoch:  7, loss: 0.037519052624702454
epoch:  8, loss: 0.036625031381845474
epoch:  9, loss: 0.036164239048957825
epoch:  10, loss: 0.035924267023801804
epoch:  11, loss: 0.03579651936888695
epoch:  12, loss: 0.035725437104701996
epoch:  13, loss: 0.035683415830135345
epoch:  14, loss: 0.0356389544904232
epoch:  15, loss: 0.03556552901864052
epoch:  16, loss: 0.03554972633719444
epoch:  17, loss: 0.035435549914836884
epoch:  18, loss: 0.03537193313241005
epoch:  19, loss: 0.03533386066555977
epoch:  20, loss: 0.03529404476284981
epoch:  21, loss: 0.0352342464029789
epoch:  22, loss: 0.035200562328100204
epoch:  23, loss: 0.03519180789589882
epoch:  24, loss: 0.035134948790073395
epoch:  25, loss: 0.03510304167866707
epoch:  26, l

In [32]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7248637080192566
Test metrics:  R2 = 0.7116726040840149


In [33]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    lr_init="BB2",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.13509082794189453
epoch:  1, loss: 0.09310416132211685
epoch:  2, loss: 0.06932585686445236
epoch:  3, loss: 0.055567070841789246
epoch:  4, loss: 0.04749790579080582
epoch:  5, loss: 0.042723339051008224
epoch:  6, loss: 0.039880987256765366
epoch:  7, loss: 0.03818157687783241
epoch:  8, loss: 0.03716231882572174
epoch:  9, loss: 0.036549534648656845
epoch:  10, loss: 0.0361802838742733
epoch:  11, loss: 0.03595732897520065
epoch:  12, loss: 0.035822413861751556
epoch:  13, loss: 0.03574056550860405
epoch:  14, loss: 0.035690732300281525
epoch:  15, loss: 0.03566021844744682
epoch:  16, loss: 0.035641368478536606
epoch:  17, loss: 0.03562956675887108
epoch:  18, loss: 0.035627346485853195
epoch:  19, loss: 0.03561658039689064
epoch:  20, loss: 0.03561323508620262
epoch:  21, loss: 0.0356033518910408
epoch:  22, loss: 0.035599108785390854
epoch:  23, loss: 0.035597916692495346
epoch:  24, loss: 0.03558476269245148
epoch:  25, loss: 0.03558256849646568
epoch:  26, lo

In [34]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8230136632919312
Test metrics:  R2 = 0.7822859287261963


In [35]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="quadratic",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5200539827346802
epoch:  1, loss: 0.28519394993782043
epoch:  2, loss: 0.09169210493564606
epoch:  3, loss: 0.03633046895265579
epoch:  4, loss: 0.03547497093677521
epoch:  5, loss: 0.03545723482966423
epoch:  6, loss: 0.035435035824775696
epoch:  7, loss: 0.03538697212934494
epoch:  8, loss: 0.03532455489039421
epoch:  9, loss: 0.03524455055594444
epoch:  10, loss: 0.03513343632221222
epoch:  11, loss: 0.0349988155066967
epoch:  12, loss: 0.03493073955178261
epoch:  13, loss: 0.03483164310455322
epoch:  14, loss: 0.0347125418484211
epoch:  15, loss: 0.03465573862195015
epoch:  16, loss: 0.03455091640353203
epoch:  17, loss: 0.034361958503723145
epoch:  18, loss: 0.03416506573557854
epoch:  19, loss: 0.033990293741226196
epoch:  20, loss: 0.03395925089716911
epoch:  21, loss: 0.033765848726034164
epoch:  22, loss: 0.03369998559355736
epoch:  23, loss: 0.03362453356385231
epoch:  24, loss: 0.03346659243106842
epoch:  25, loss: 0.033262159675359726
epoch:  26, loss: 0.

In [36]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7122867107391357
Test metrics:  R2 = 0.7000678181648254


In [37]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="lipschitz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.10077954083681107
epoch:  1, loss: 0.06863167881965637
epoch:  2, loss: 0.05221296101808548
epoch:  3, loss: 0.04396848753094673
epoch:  4, loss: 0.03989240899682045
epoch:  5, loss: 0.03790237009525299
epoch:  6, loss: 0.036939892917871475
epoch:  7, loss: 0.0364767387509346
epoch:  8, loss: 0.036253832280635834
epoch:  9, loss: 0.03614570200443268
epoch:  10, loss: 0.03609214723110199
epoch:  11, loss: 0.03606444597244263
epoch:  12, loss: 0.03604898974299431
epoch:  13, loss: 0.03603920340538025
epoch:  14, loss: 0.03601161018013954
epoch:  15, loss: 0.03599650785326958
epoch:  16, loss: 0.03599190711975098
epoch:  17, loss: 0.03596459701657295
epoch:  18, loss: 0.03594975546002388
epoch:  19, loss: 0.035948507487773895
epoch:  20, loss: 0.03592091426253319
epoch:  21, loss: 0.035906072705984116
epoch:  22, loss: 0.035897206515073776
epoch:  23, loss: 0.035878848284482956
epoch:  24, loss: 0.03586406260728836
epoch:  25, loss: 0.03585527837276459
epoch:  26, loss:

In [38]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7324620485305786
Test metrics:  R2 = 0.7135181427001953


In [39]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    lr_init="keep",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.10772253572940826
epoch:  1, loss: 0.06432637572288513
epoch:  2, loss: 0.046410370618104935
epoch:  3, loss: 0.039185140281915665
epoch:  4, loss: 0.036304913461208344
epoch:  5, loss: 0.03515659645199776
epoch:  6, loss: 0.034696128219366074
epoch:  7, loss: 0.03450780361890793
epoch:  8, loss: 0.03442678600549698
epoch:  9, loss: 0.03438795357942581
epoch:  10, loss: 0.0343656949698925
epoch:  11, loss: 0.03434992954134941
epoch:  12, loss: 0.03433680534362793
epoch:  13, loss: 0.03432464599609375
epoch:  14, loss: 0.034312594681978226
epoch:  15, loss: 0.034300923347473145
epoch:  16, loss: 0.03428925201296806
epoch:  17, loss: 0.03427765145897865
epoch:  18, loss: 0.034266043454408646
epoch:  19, loss: 0.03425418958067894
epoch:  20, loss: 0.03424200043082237
epoch:  21, loss: 0.03422953933477402
epoch:  22, loss: 0.0342167466878891
epoch:  23, loss: 0.034203462302684784
epoch:  24, loss: 0.034190014004707336
epoch:  25, loss: 0.0341765433549881
epoch:  26, loss

In [40]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6982423067092896
Test metrics:  R2 = 0.6925662159919739
